# 29. Tensor Parallelism Sim | Tensor 并行模拟

**难度：** Hard | **环境：** CPU-first | **标签：** `并行通信`, `Tensor Parallelism`, `通信` | **目标人群：** 并行通信学习者

---

## 本节导读

ZeRO 主要切训练状态，Pipeline 主要切模型层；但很多大模型里的单个 Linear 本身就很大，即使按层切完，某一层的矩阵乘法仍可能成为显存和计算瓶颈。这时需要把一个权重矩阵沿维度拆到多张卡上共同计算。

Tensor Parallelism 的核心就是按张量维度切分 Linear：Column Parallel 切输出维度，通常需要 All-Gather；Row Parallel 切输入维度，通常需要 All-Reduce。本节用纯 PyTorch 模拟这两种切法，重点看清切分方向、局部输出和通信模式如何对应。完成后，你应该能区分 ZeRO、Pipeline 和 Tensor Parallelism 各自“切的对象”是什么。

**关键词：** `Tensor Parallelism`, `Column Parallel`, `Row Parallel`

---


## 前置阅读

**导语：** 进入本节前，先能回答 ZeRO 切分训练状态、Pipeline 切分模型层分别解决什么问题，再观察 Tensor Parallelism 如何切分单个矩阵。

- [27. ZeRO Optimizer Sim | ZeRO 优化器模拟](../02_PyTorch_Algorithms/27_ZeRO_Optimizer_Sim.ipynb)
- [28. Pipeline Parallelism MicroBatch | Pipeline 并行微批次](../02_PyTorch_Algorithms/28_Pipeline_Parallelism_MicroBatch.ipynb)
- [P1: 26. Parallel Strategy Decision Framework | 并行策略决策框架](../01_Hardware_Math_and_Systems/26_Parallel_Strategy_Decision_Framework.ipynb)


---

### Step 1：理解 Tensor Parallel 的两种切分

假设输入 $X$ 形状为 `(batch, in_dim)`，权重 $A$ 形状为 `(in_dim, out_dim)`，经过线性层变为 $Y = XA$，形状 `(batch, out_dim)`。

> **Column Parallel (列切分)：切分 $A$ 的列 (输出维度)**
> 1. $A$ 被竖着切成左右两块 $A_1, A_2$ 分别放到 GPU 0 和 1。
> 2. GPU 0 计算 $Y_1 = X A_1$，GPU 1 计算 $Y_2 = X A_2$。
> 3. **通信：** 各自算完后，通过 `All-Gather`，把左右结果拼起来，得到完整的 $Y = [Y_1, Y_2]$。
> *适用场景：MLP 的第一个全连接层（扩大隐藏维度时）。*

> **Row Parallel (行切分)：切分 $A$ 的行 (输入维度)**
> 1. $A$ 被横着切成上下两块 $A_1, A_2$ 分别放到 GPU 0 和 1。
> 2. 输入 $X$ 也要沿着特征维度切成左右两半 $X_1, X_2$ 给不同的卡。
> 3. GPU 0 计算 $Y_1 = X_1 A_1$，GPU 1 计算 $Y_2 = X_2 A_2$。
> 4. **通信：** 完整的结果其实是两者的加和：$Y = Y_1 + Y_2$。所以需要做一次 `All-Reduce (Sum)`。
> *适用场景：MLP 的第二个全连接层（缩回原始维度时）。*

**精妙之处**：如果把 Column Parallel 放前面，Row Parallel 放后面，中间甚至可以省掉一次通信。

![Tensor Parallelism：权重切分决定通信位置](../docs/public/02_PyTorch_Algorithms/29_tensor_parallel_split.svg)


### Step 2：沿计算图安排通信路径

在一个两层的前馈网络 $Y = X \cdot W_1 \cdot W_2$ 中：
- 我们将 $W_1$ 按列切分（Column Parallel），得到两块。计算后各个 GPU 得到不完整的部分输出矩阵。
- 紧接着，将 $W_2$ 按行切分（Row Parallel），利用刚才的部分输出分别与之相乘。
- 最后，所有 GPU 执行一次 `All-Reduce` 聚合结果。这样在两层神经网络中，只产生了一次通信开销。

![Tensor Parallelism：切分方向决定通信算子](../docs/public/02_PyTorch_Algorithms/29_tensor_parallel_communication.svg)


### Step 3：比较切分收益与通信代价
Column Parallel 将局部输出留给下一层的 Row Parallel，可以减少中间激活的同步；但最终仍需在正确的位置聚合结果。实际 TP 选型还要检查张量维度是否可整除、互联是否足够快，以及 collective 是否落在关键路径上。

| 路径 | 切分对象 | 局部输出如何恢复 | 常见位置 | 主要代价 |
|---|---|---|---|---|
| Column Parallel | 权重输出维 | All-Gather / 拼接 | MLP 扩维层 | 输入复制与输出聚合 |
| Row Parallel | 输入与权重输入维 | All-Reduce / 求和 | MLP 缩维层 | 输出归约 |
| Column → Row 组合 | 中间激活保持分片 | 末端统一归约 | 两层 MLP | 依赖布局匹配，减少中间同步 |


### Step 4：CPU 实现——验证 Column 与 Row Parallel 等价性

下面把两条通信路径落实为 CPU 模拟：学习者分别完成列切分与行切分的局部计算，再用拼接或求和恢复单卡矩阵乘法的结果。题目区只保留实现机制所需的变量，测试区负责验证数值等价性和形状契约。

| 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
|---|---|---|---|
| `column_parallel_linear` | 沿输出维切分权重，计算局部输出，再拼接 | `out_dim` 可被 rank 数整除；拼接顺序与原输出一致 | Column 路径与 `X @ A` 的数值和形状一致 |
| `row_parallel_linear` | 沿输入维切分输入和权重，计算局部结果，再求和 | `in_dim` 可被 rank 数整除；各局部块维度匹配 | Row 路径与 `X @ A` 的数值和形状一致 |
| shape contract | 检查切分、局部结果与恢复结果的形状 | 不允许隐式改变 batch 或输出维 | Column / Row / 总体形状测试 |


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

In [ ]:
def tensor_parallel_column_sim(X: torch.Tensor, A: torch.Tensor, num_gpus: int = 2):
    """
    模拟 Column Parallel Linear: Y = X @ A
    将权重 A 沿列 (输出特征维度) 切分，分布到不同的 GPU 上计算，最后拼接。
    
    参数:
    X: 形状 (batch, in_features)
    A: 形状 (in_features, out_features)
    """
    in_features, out_features = A.shape
    assert out_features % num_gpus == 0, "输出维度必须能被 GPU 数量整除"
    
    chunk_size = out_features // num_gpus
    
    # 1. 模拟将权重加载到不同 GPU 的显存中
    # a_chunks 是一个列表，代表各 GPU 本地保存的权重分片
    a_chunks = []
    for i in range(num_gpus):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size
        # ==========================================
        # TODO 1（Column）：沿输出维切分权重 A
        # 变量提示（每个变量各占一行）：
        # a_chunk = A[:, start_idx:end_idx]
        # ==========================================
        # a_chunk = ???
        a_chunks.append(a_chunk)
        
    # 2. 模拟各 GPU 并行前向计算
    # 在真实环境中，X 会被广播到所有 GPU (因为是列切分，输入不需要切)
    y_chunks = []
    for i in range(num_gpus):
        # ==========================================
        # TODO 2（Column）：使用本地权重完成局部矩阵乘法
        # 变量提示（每个变量各占一行）：
        # a_local = a_chunks[i]
        # y_local = X @ a_local
        y_chunks.append(y_local)
        
    # 3. 模拟 All-Gather 通信操作
    # ==========================================
    # TODO 3（Column）：沿输出维拼接局部结果，模拟 All-Gather
    # 变量提示（每个变量各占一行）：
    # Y_tp = torch.cat(y_chunks, dim=-1)
    # ==========================================
    # Y_tp = ???
    return Y_tp


def tensor_parallel_row_sim(X: torch.Tensor, A: torch.Tensor, num_gpus: int = 2):
    """
    模拟 Row Parallel Linear: Y = X @ A
    将权重 A 沿行 (输入特征维度) 切分，输入 X 也同步切分，最后将各卡输出求和。
    
    参数:
    X: 形状 (batch, in_features)
    A: 形状 (in_features, out_features)
    """
    in_features, out_features = A.shape
    assert in_features % num_gpus == 0, "输入维度必须能被 GPU 数量整除"
    
    chunk_size = in_features // num_gpus
    
    # 1. 模拟将输入和权重切分给不同 GPU 的显存中
    x_chunks = []
    a_chunks = []
    for i in range(num_gpus):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size
        # ==========================================
        # TODO 4（Row）：同步切分输入与权重
        # 变量提示（每个变量各占一行）：
        # x_chunk = X[:, start_idx:end_idx]
        # a_chunk = A[start_idx:end_idx, :]
        # ==========================================
        # a_chunk = ???
        # x_chunk = ???
        a_chunks.append(a_chunk)
        x_chunks.append(x_chunk)
        
    # 2. 模拟各 GPU 并行前向计算
    y_outputs = []
    for i in range(num_gpus):
        # ==========================================
        # TODO 5（Row）：使用本地输入和权重完成局部矩阵乘法
        # 变量提示（每个变量各占一行）：
        # x_local = x_chunks[i]
        # a_local = a_chunks[i]
        # y_local = x_local @ a_local
        y_outputs.append(y_local)
        
    # 3. 模拟 All-Reduce (Sum)
    # ==========================================
    # TODO 6（Row）：按元素求和局部结果，模拟 All-Reduce
    # 变量提示（每个变量各占一行）：
    # Y_tp = torch.stack(y_outputs, dim=0).sum(dim=0)
    # ==========================================
    # Y_tp = ???
    return Y_tp


In [ ]:
# 测试你的实现
def test_tensor_parallel():
    # 回归验证 Column/Row TP 与单卡矩阵乘法的数值等价性。
    try:
        torch.manual_seed(42)
        batch_size = 4
        in_dim = 16
        out_dim = 32
        
        # 原始数据
        X = torch.randn(batch_size, in_dim)
        A = torch.randn(in_dim, out_dim)
        
        # 1. 单卡全量计算作为 Ground Truth
        Y_ref = X @ A
        
        # 2. 模拟 2 张卡的 Column Parallel
        Y_col = tensor_parallel_column_sim(X, A, num_gpus=2)
        diff_col = torch.max(torch.abs(Y_ref - Y_col))
        print(f"Column Parallel 最大误差: {diff_col.item():.6e}")
        assert Y_col.shape == Y_ref.shape, "Column Parallel 输出形状错误！"
        assert diff_col < 1e-5, "Column Parallel 模拟结果与单卡全量计算不一致！"
        
        # 3. 模拟 2 张卡的 Row Parallel
        Y_row = tensor_parallel_row_sim(X, A, num_gpus=2)
        diff_row = torch.max(torch.abs(Y_ref - Y_row))
        print(f"Row Parallel 最大误差: {diff_row.item():.6e}")
        assert Y_row.shape == Y_ref.shape, "Row Parallel 输出形状错误！"
        assert diff_row < 1e-5, "Row Parallel 模拟结果与单卡全量计算不一致！"
        
        # 4. 维度约束检查
        try:
            tensor_parallel_column_sim(X, A[:, :30], num_gpus=2)
            raise AssertionError("Column Parallel 应该要求输出维度可整除")
        except AssertionError:
            pass
        
        try:
            tensor_parallel_row_sim(X[:, :15], A[:15], num_gpus=2)
            raise AssertionError("Row Parallel 应该要求输入维度可整除")
        except AssertionError:
            pass
        
        print("✅ Column Parallel (列切分) 矩阵计算与拼接逻辑正确！")
        print("✅ Row Parallel (行切分) 矩阵计算与求和逻辑正确！")
        print("掌握了 Megatron-LM 的核心张量切分思路，单卡装不下的大规模参数量再也不是问题。")
        
    except NotImplementedError:
        print("请先完成 TODO 部分的代码！")
        raise
    except (AttributeError, NameError, TypeError, ValueError, AssertionError, RuntimeError) as e:
        if isinstance(e, AttributeError):
            print("代码未完成，无法找到必要的属性")
        elif isinstance(e, NameError):
            print("代码可能未完成，导致了变量未定义")
        elif isinstance(e, TypeError):
            print("代码可能未完成，导致了类型错误")
        elif isinstance(e, ValueError):
            print("代码可能未完成，导致了张量维度错误")
        elif isinstance(e, AssertionError):
            print("代码可能未完成，导致了断言失败")
        else:
            print("代码可能未完成，导致了运行时错误")
        raise NotImplementedError("请先完成 TODO 部分的代码！") from e
    except Exception as e:
        print(f"❌ 测试失败: {e}")
        raise

def test_column_parallel_contract():
    # 验证 Column Parallel 的局部切分与输出拼接契约。
    torch.manual_seed(42)
    X = torch.randn(2, 8)
    A = torch.randn(8, 12)
    assert torch.allclose(tensor_parallel_column_sim(X, A, 2), X @ A)

def test_row_parallel_contract():
    # 验证 Row Parallel 的局部乘法与 All-Reduce 求和契约。
    torch.manual_seed(42)
    X = torch.randn(2, 8)
    A = torch.randn(8, 12)
    assert torch.allclose(tensor_parallel_row_sim(X, A, 2), X @ A, atol=1e-5)

def test_tensor_parallel_shape_contract():
    # 验证两种 TP 路径都恢复预期输出形状。
    X = torch.randn(2, 8)
    A = torch.randn(8, 12)
    assert tensor_parallel_column_sim(X, A, 2).shape == (2, 12)
    assert tensor_parallel_row_sim(X, A, 2).shape == (2, 12)

test_column_parallel_contract()
test_row_parallel_contract()
test_tensor_parallel_shape_contract()
test_tensor_parallel()


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果你正在 Colab 中运行，并且遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---

## 参考代码与解析

### 代码


In [ ]:
def tensor_parallel_column_sim(X, A, num_gpus):
    # TODO 1: 权重切分 (Scatter)
    in_features, out_features = A.shape
    chunk_size = out_features // num_gpus
    a_chunks = []
    for i in range(num_gpus):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size
        a_chunk = A[:, start_idx:end_idx]
        a_chunks.append(a_chunk)
    
    # TODO 2: 独立计算 (Local MatMul)
    y_chunks = []
    for i in range(num_gpus):
        a_local = a_chunks[i]
        y_local = X @ a_local
        y_chunks.append(y_local)
        
    # TODO 3: 结果合并 (All-Gather)
    Y_tp = torch.cat(y_chunks, dim=-1)
    return Y_tp


def tensor_parallel_row_sim(X, A, num_gpus):
    # TODO 4: 输入和权重切分 (Scatter)
    in_features, out_features = A.shape
    chunk_size = in_features // num_gpus
    x_chunks = []
    a_chunks = []
    for i in range(num_gpus):
        start_idx = i * chunk_size
        end_idx = start_idx + chunk_size
        x_chunk = X[:, start_idx:end_idx]
        a_chunk = A[start_idx:end_idx, :]
        x_chunks.append(x_chunk)
        a_chunks.append(a_chunk)
    
    # TODO 5: 独立计算 (Local MatMul)
    y_chunks = []
    for i in range(num_gpus):
        x_local = x_chunks[i]
        a_local = a_chunks[i]
        y_local = x_local @ a_local
        y_chunks.append(y_local)
        
    # TODO 6: 结果求和 (All-Reduce)
    Y_tp = torch.stack(y_chunks, dim=0).sum(dim=0)
    return Y_tp


### 解析

**1. TODO 1: Column Parallel 的权重切分 (Scatter)**
- **实现方式**：`a_chunks = torch.chunk(A, num_gpus, dim=1)`
- **关键点**：将权重沿输出特征维度切分，每张卡只保存一部分列分片
- **技术细节**：Column Parallel 的核心是切分权重，而不是切分输入

**2. TODO 2: Column Parallel 的独立计算 (Local MatMul)**
- **实现方式**：`y_local = X @ a_local`
- **关键点**：输入 `X` 会被广播到所有卡，每张卡独立计算自己的输出分片
- **技术细节**：各卡计算得到的是输出特征的一部分，不是完整输出

**3. TODO 3: Column Parallel 的结果合并 (All-Gather)**
- **实现方式**：`Y_tp = torch.cat(y_chunks, dim=-1)`
- **关键点**：将各卡输出沿特征维拼接，恢复完整输出
- **技术细节**：这一步对应张量并行中的 All-Gather / 拼接操作

**4. TODO 4: Row Parallel 的输入和权重切分 (Scatter)**
- **实现方式**：`x_chunks = torch.chunk(X, num_gpus, dim=1)`，`a_chunks = torch.chunk(A, num_gpus, dim=0)`
- **关键点**：Row Parallel 同时切输入和权重，分别对应输入特征和权重行
- **技术细节**：这一步是 Row Parallel 与 Column Parallel 的核心差异之一

**5. TODO 5: Row Parallel 的独立计算 (Local MatMul)**
- **实现方式**：`y_local = x_local @ a_local`
- **关键点**：每张卡只计算自己分片对应的部分输出
- **技术细节**：各卡结果是“部分和”，还不能直接作为最终输出

**6. TODO 6: Row Parallel 的结果求和 (All-Reduce)**
- **实现方式**：`Y_tp = torch.stack(y_chunks, dim=0).sum(dim=0)`
- **关键点**：将各卡结果按元素相加，恢复完整输出
- **技术细节**：这一步对应张量并行中的 All-Reduce (Sum)

**工程要点**
- **通信特点**：Column Parallel 需要广播输入、合并输出；Row Parallel 需要切分输入、最后求和
- **适用场景**：Column Parallel 更适合扩维层，Row Parallel 更适合缩维层
- **组合方式**：在两层 MLP 中常见 Column -> Row 的组合，可以减少中间通信


### Step 5（可选）：GPU Tensor Parallel 通信探针

#### 5.1 环境与固定 workload

使用两张 GPU，固定 dtype、矩阵形状、warmup 和 repeats；单卡矩阵乘法是 baseline，Column / Row 的 collective 路径是 candidate。GPU 实验默认关闭，学习者只有在具备多卡环境时再开启。


In [ ]:
# 5.1：默认关闭；真实 TP collective 探针至少需要两张 CUDA GPU。
RUN_GPU_EXPERIMENT = False
WORLD_SIZE = 2
RESULT_PATH = 'benchmarks/results/29_tensor_parallel_smoke.json'
print({'run': RUN_GPU_EXPERIMENT, 'world_size': WORLD_SIZE, 'result': RESULT_PATH})


#### 5.2 配置与执行

默认关闭。开启后用 `torchrun` 初始化 NCCL，并依次执行 `all_gather` 与 `all_reduce`；该探针用于观察 TP 通信路径是否可运行，不等同于完整 Megatron 模型吞吐。


In [ ]:
# 5.2：执行 all_gather → all_reduce 探针；默认关闭。
if RUN_GPU_EXPERIMENT:
    import subprocess
    command = ['torchrun', '--standalone', '--nproc_per_node', str(WORLD_SIZE), 'tools/run_distributed_smoke.py', '--project', '29', '--output', RESULT_PATH]
    subprocess.run(command, check=True)
else:
    print('GPU collective 探针默认关闭。')


#### 5.3 读取结果、解释指标与形成决策

结果记录 world size、collective、payload、耗时、硬件、failure 和 evidence level。将数值等价性与 collective 耗时分开解读：前者证明切分正确，后者说明当前互联上的通信代价；完整训练吞吐仍需在 79 的项目 benchmark 中验证。


In [ ]:
# 5.3：读取通信探针 JSON；它只证明 TP collective 路径。
import json
from pathlib import Path
if Path(RESULT_PATH).exists():
    result = json.loads(Path(RESULT_PATH).read_text())
    required = {'operation', 'world_size', 'hardware', 'elapsed_ms', 'evidence_level', 'failure'}
    missing = required - set(result)
    if missing:
        raise ValueError(f'结果 JSON 缺少字段：{sorted(missing)}')
    print(result)
else:
    print(f'尚无 TP 探针结果：{RESULT_PATH}')


## 相关阅读

Tensor Parallelism 的关键不是切分本身，而是切分后计算和通信如何交替。可以继续阅读 Megatron-LM 和分布式基准项目。

- [Megatron-LM 原论文](https://arxiv.org/abs/2104.04473)
- [Megatron-LM 开源仓库](https://github.com/NVIDIA/Megatron-LM)
- [P1: CUDA Stream 与异步执行](../01_Hardware_Math_and_Systems/17_CUDA_Stream_and_Asynchrony.ipynb)
- [P1: 通信调度优化](../01_Hardware_Math_and_Systems/27_Communication_Scheduling_Optimization.ipynb)
- [79. 分布式并行基准](../02_PyTorch_Algorithms/79_Distributed_Parallel_Benchmark.ipynb)
